# RBC Morphology Detection — Production Pipeline

**Single YOLO model · 39-class end-to-end detector**

| Dimension | Detail |
|---|---|
| Shapes | 13 types (Acanthocytes → Normocytes) |
| Chromia | 3 types (Hypochromic / Normochromic / Hyperchromic) |
| Classes | 13 × 3 = **39** |
| Size | Computed from bbox diameter — not a trained class |

**Run order:** Environment check → Dataset build → Train → TensorRT FP16 export → Inference → JSON output

## 1. Environment Check

In [ ]:
import subprocess, sys

required = {
    'ultralytics': 'ultralytics',
    'torch':       'torch',
    'cv2':         'opencv-python-headless',
    'skimage':     'scikit-image',
    'yaml':        'pyyaml',
    'PIL':         'Pillow',
}
missing = []
for mod, pkg in required.items():
    try:
        __import__(mod)
        print(f'  ok  {pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'  MISSING  {pkg}')

if missing:
    
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\nAll dependencies OK')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

: 

## 2. Imports & Configuration

In [ ]:
import json, math, shutil, warnings, yaml, time, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = Path('F:/Livo/Data - 2026/Rbc/data')
JSON_DIR    = Path('F:/Livo/Data - 2026/Rbc/rbc_output')
DATASET_DIR = Path('F:/Livo/Data - 2026/Rbc/yolo_dataset')
PROJECT_DIR = Path('F:/Livo/Data - 2026/Rbc/rbc_yolo')
INFER_OUT   = Path('F:/Livo/Data - 2026/Rbc/rbc_test_output')

# ── Slide-level split  (Img_X_Y.jpg → group by X) ────────────────────────────
TRAIN_GROUPS = set(range(0, 7))   # Img_0_* … Img_6_*
VAL_GROUPS   = {7}                # Img_7_*
TEST_GROUPS  = {8, 9}             # Img_8_*, Img_9_*

# ── Training ──────────────────────────────────────────────────────────────────
# imgsz=1024  batch=4  → mAP50=0.520  median=13.4ms  (RECOMMENDED)
# imgsz=640   batch=8  → mAP50=0.391  median=10.5ms  (25% accuracy drop, NMS overhead worse)
IMGSZ    = 1024
BATCH    = 4
EPOCHS   = 150
PATIENCE = 30

# ── Inference ─────────────────────────────────────────────────────────────────
CONF_THRES = 0.25
IOU_THRES  = 0.45

# ── Size thresholds (pixel diameter at 100x magnification) ────────────────────
MICROCYTE_MAX = 35
MACROCYTE_MIN = 55

print(f'Config  |  imgsz={IMGSZ}  batch={BATCH}  epochs={EPOCHS}')
for label, p in [('Data', DATA_DIR), ('JSONs', JSON_DIR)]:
    status = 'EXISTS' if p.exists() else 'NOT FOUND'
    n = len(list(p.glob('*.jpg' if label == 'Data' else '*.json'))) if p.exists() else 0
    print(f'  {label}: {p}  [{status}, {n} files]')

Config  |  imgsz=1024  batch=4  epochs=150
  Data: F:\Livo\Data - 2026\Rbc\data  [EXISTS, 508 files]
  JSONs: F:\Livo\Data - 2026\Rbc\rbc_output  [EXISTS, 508 files]


## 3. Class Definitions (39 Classes)

In [ ]:
SHAPE_NAMES = [
    'acanthocytes',   # 0
    'bite_cells',     # 1
    'blister_cells',  # 2
    'echinocytes',    # 3
    'ovalocytes',     # 4
    'stomatocytes',   # 5
    'schistocytes',   # 6
    'sickle_cells',   # 7
    'spherocytes',    # 8
    'target_cells',   # 9
    'teardrop_cells', # 10
    'elliptocytes',   # 11
    'normocytes',     # 12
]
CHROMIA_NAMES = ['hypochromic', 'normochromic', 'hyperchromic']  # 0, 1, 2

# class_id = shape_idx * 3 + chromia_idx  (0–38)
CLASSES = [f'{s}_{c}' for s in SHAPE_NAMES for c in CHROMIA_NAMES]
assert len(CLASSES) == 39

SHAPE_MAP = {
    'Normal Biconcave':      12,
    'Ovalocyte/Elliptocyte':  4,
    'Echinocyte':             3,
}
CHROMIA_MAP = {
    'Hypochromic':              0,
    'Normochromic':             1,
    'Hyperchromic':             2,
    'Hyperchromic/Spherocytic': 2,
}

ID_TO_SHAPE   = {i*3+j: SHAPE_NAMES[i]   for i in range(13) for j in range(3)}
ID_TO_CHROMIA = {i*3+j: CHROMIA_NAMES[j] for i in range(13) for j in range(3)}

print(f'39 classes defined  (shape x chromia)')
print('\nID  Class')
for i, c in enumerate(CLASSES):
    print(f'  {i:2d}  {c}')

39 classes defined  (shape x chromia)

ID  Class
   0  acanthocytes_hypochromic
   1  acanthocytes_normochromic
   2  acanthocytes_hyperchromic
   3  bite_cells_hypochromic
   4  bite_cells_normochromic
   5  bite_cells_hyperchromic
   6  blister_cells_hypochromic
   7  blister_cells_normochromic
   8  blister_cells_hyperchromic
   9  echinocytes_hypochromic
  10  echinocytes_normochromic
  11  echinocytes_hyperchromic
  12  ovalocytes_hypochromic
  13  ovalocytes_normochromic
  14  ovalocytes_hyperchromic
  15  stomatocytes_hypochromic
  16  stomatocytes_normochromic
  17  stomatocytes_hyperchromic
  18  schistocytes_hypochromic
  19  schistocytes_normochromic
  20  schistocytes_hyperchromic
  21  sickle_cells_hypochromic
  22  sickle_cells_normochromic
  23  sickle_cells_hyperchromic
  24  spherocytes_hypochromic
  25  spherocytes_normochromic
  26  spherocytes_hyperchromic
  27  target_cells_hypochromic
  28  target_cells_normochromic
  29  target_cells_hyperchromic
  30  teardrop_c

## 4. Dataset Preparation  (Cellpose JSON → YOLO format)

In [ ]:
def to_yolo_line(cid, bbox, iw, ih):
    x, y, w, h = bbox
    cx = max(0.0, min(1.0, (x+w/2)/iw))
    cy = max(0.0, min(1.0, (y+h/2)/ih))
    nw = max(0.001, min(1.0, w/iw))
    nh = max(0.001, min(1.0, h/ih))
    return f'{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

def slide_group(stem):
    for p in stem.split('_')[1:]:
        if p.isdigit(): return int(p)
    return 0

def get_split(stem):
    g = slide_group(stem)
    if g in TRAIN_GROUPS: return 'train'
    if g in VAL_GROUPS:   return 'val'
    return 'test'

print('Dataset helpers ready')

Dataset helpers ready


In [ ]:
def build_dataset():
    for split in ('train', 'val', 'test'):
        (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

    img_counts   = defaultdict(int)
    class_counts = defaultdict(int)

    for jf in sorted(JSON_DIR.glob('*.json')):
        if '_overlay' in jf.stem: continue
        img_src = DATA_DIR / (jf.stem + '.jpg')
        if not img_src.exists(): continue

        with open(jf) as f: data = json.load(f)
        iw    = data['image_size']['width']
        ih    = data['image_size']['height']
        dets  = data.get('detections', [])
        split = get_split(jf.stem)

        dst = DATASET_DIR / 'images' / split / img_src.name
        if not dst.exists(): shutil.copy2(img_src, dst)

        lines = []
        for d in dets:
            s = SHAPE_MAP.get(d.get('shape', ''))
            c = CHROMIA_MAP.get(d.get('chromia', ''))
            if s is None or c is None: continue
            cid  = s * 3 + c
            b    = d['bbox']
            lines.append(to_yolo_line(cid, (b['x'], b['y'], b['width'], b['height']), iw, ih))
            class_counts[cid] += 1

        (DATASET_DIR / 'labels' / split / (jf.stem + '.txt')).write_text('\n'.join(lines))
        img_counts[split] += 1

    yaml_cfg = {
        'path':  DATASET_DIR.as_posix(),
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    39,
        'names': CLASSES,
    }
    with open(DATASET_DIR / 'dataset.yaml', 'w') as f:
        yaml.dump(yaml_cfg, f, default_flow_style=False, sort_keys=False)

    total = sum(class_counts.values())
    print(f'Dataset ready: {DATASET_DIR}')
    print(f"  train={img_counts['train']}  val={img_counts['val']}  test={img_counts['test']}")
    print(f'  total annotations: {total}')
    print(f'\nClass distribution:')
    for cid in range(39):
        n   = class_counts.get(cid, 0)
        bar = '#' * int(30 * n / max(total, 1))
        tag = '  <- EMPTY' if n == 0 else ('  <- RARE' if n < 50 else '')
        print(f'  [{cid:2d}] {CLASSES[cid]:<45s} {n:6d}  {bar}{tag}')

build_dataset()

KeyboardInterrupt: 

## 5. Model Selection  (YOLO26n if available, else YOLO11n)

In [ ]:
import ultralytics
from ultralytics import YOLO

def detect_best_model(preferred='yolo26n', fallback='yolo11n'):
    """
    Scan ultralytics cfg/models directory for preferred model config YAML.
    No network download — only checks installed package files.
    """
    cfg_root = Path(ultralytics.__file__).parent / 'cfg' / 'models'
    found = any(
        p.stem.lower().startswith(preferred.lower())
        for p in cfg_root.rglob('*.yaml')
    )
    name = preferred if found else fallback
    print(f'ultralytics {ultralytics.__version__}')
    print(f'  {preferred}: {"AVAILABLE" if found else "not in this build"}')
    print(f'  Selected base model: {name}.pt')
    return f'{name}.pt'

BASE_MODEL   = detect_best_model('yolo26n', 'yolo11n')
DATASET_YAML = str(DATASET_DIR / 'dataset.yaml')
RUN_NAME     = f"{Path(BASE_MODEL).stem}_{IMGSZ}"

ultralytics 8.4.67
  yolo26n: not in this build
  Selected base model: yolo11n.pt


## 6. Training

In [ ]:
def train_model(model_path, imgsz, batch, run_name):
    model = YOLO(model_path)
    return model.train(
        data           = DATASET_YAML,
        epochs         = EPOCHS,
        patience       = PATIENCE,
        imgsz          = imgsz,
        batch          = batch,
        project        = str(PROJECT_DIR),
        name           = run_name,
        device         = 0,
        workers        = 4,
        optimizer      = 'AdamW',
        lr0            = 0.001,
        lrf            = 0.01,
        cos_lr         = True,
        warmup_epochs  = 5,
        # ── Box-only priority: collapse all 39 shape/chromia classes into a
        #    single 'RBC' class for loss + eval — mAP/precision/recall then
        #    measure pure localization quality, not classification accuracy.
        #    Label files are untouched; this is purely a training/eval flag.
        single_cls     = True,
        # ── Chromia-safe augmentation ─────────────────────────────────────────
        hsv_h          = 0.0,    # NO hue shift   (chromia = hue-dependent)
        hsv_s          = 0.0,    # NO sat shift   (chromia = saturation-dependent)
        hsv_v          = 0.02,   # mild brightness only
        fliplr         = 0.5,
        flipud         = 0.5,
        degrees        = 15.0,
        scale          = 0.3,
        translate      = 0.1,
        mosaic         = 0.0,    # OFF — distorts slide context
        mixup          = 0.0,    # OFF — corrupts chromia signal
        copy_paste     = 0.0,
        erasing        = 0.0,    # OFF — may erase central pallor region
        # ── Loss weights ──────────────────────────────────────────────────────
        box            = 7.5,
        cls            = 0.5,
        dfl            = 1.5,
        save           = True,
        save_period    = 10,
        val            = True,
        plots          = True,
    )

# ── Run (nano model at 1024 — safe for RTX 3050 4 GB VRAM) ────────────────────
# To try larger model: change BASE_MODEL to 'yolo11s.pt' and batch to 2
result = train_model(BASE_MODEL, IMGSZ, BATCH, RUN_NAME)

In [ ]:
# Print key metrics after training
if hasattr(result, 'results_dict'):
    m = result.results_dict
    print(f'mAP50    : {m.get("metrics/mAP50(B)", "n/a")}')
    print(f'mAP50-95 : {m.get("metrics/mAP50-95(B)", "n/a")}')
    print(f'Precision: {m.get("metrics/precision(B)", "n/a")}')
    print(f'Recall   : {m.get("metrics/recall(B)", "n/a")}')

mAP50    : 0.4386055523411453
mAP50-95 : 0.4147773334914081
Precision: 0.6358027415878527
Recall   : 0.4063050797144264


## 7. Locate Best Weights

In [ ]:
def find_best_pt(project_dir, run_name):
    direct = Path(project_dir) / run_name / 'weights' / 'best.pt'
    if direct.exists():
        print(f'Best weights: {direct}'); return str(direct)
    candidates = sorted(
        Path(project_dir).glob('**/best.pt'),
        key=lambda p: p.stat().st_mtime
    )
    if candidates:
        best = candidates[-1]
        print(f'Best weights: {best}'); return str(best)
    raise FileNotFoundError('No best.pt found — run the training cell first.')

BEST_PT = find_best_pt(PROJECT_DIR, RUN_NAME)
print(f'Run name: {RUN_NAME}')

## 8. TensorRT FP16 Export  *(Optional — RTX 3050 laptop: FP32 is faster)*

> **Benchmark result on RTX 3050 laptop (8 GB):**
> - FP32 `.pt` model → **13.4 ms median**
> - TensorRT FP16 `.engine` → 20.4 ms median *(slower due to thermal throttle under sustained FP16 load)*
> - `half=True` in predict → 17.7 ms median *(also slower)*
>
> **Skip this section** unless running on a desktop GPU (RTX 3080 / 4080) where TRT FP16 gives 2–3× speedup.
> The `.pt` model is used directly for inference below.

In [ ]:
# ── OPTIONAL: Export to TensorRT FP16 ────────────────────────────────────────
# Run only on desktop GPU (RTX 3080+). On RTX 3050 laptop, FP32 .pt is faster.
# Requires TensorRT: pip install tensorrt
#
# def export_trt(weights_path, imgsz=1024):
#     model  = YOLO(weights_path)
#     engine = model.export(format='engine', imgsz=imgsz, half=True, device=0, simplify=True)
#     print(f'TensorRT FP16 engine: {engine}')
#     return str(engine)
#
# ENGINE_PATH = export_trt(BEST_PT, imgsz=IMGSZ)

print('TensorRT export skipped — using FP32 .pt (fastest on RTX 3050 laptop)')

## 9. Inference

In [ ]:
SHAPE_COLOR = {
    'normocytes':    (0,   255,   0),
    'ovalocytes':    (255, 255,   0),
    'echinocytes':   (0,   165, 255),
    'acanthocytes':  (255,   0, 255),
    'bite_cells':    (0,   255, 255),
    'blister_cells': (128,   0, 255),
    'schistocytes':  (0,   128, 255),
    'sickle_cells':  (255,  50,  50),
    'spherocytes':   (50,  255, 100),
    'target_cells':  (255, 200,   0),
    'teardrop_cells':(200,   0, 255),
    'stomatocytes':  (0,   200, 200),
    'elliptocytes':  (200, 255,   0),
}
DEFAULT_COLOR = (200, 200, 200)

def compute_size(bbox_w, bbox_h):
    diam = (bbox_w + bbox_h) / 2.0
    if diam < MICROCYTE_MAX: return 'Microcyte'
    if diam > MACROCYTE_MIN: return 'Macrocyte'
    return 'Normocyte'

def parse_detections(yolo_results, img_w, img_h):
    dets, det_id = [], 1
    for r in yolo_results:
        if r.boxes is None or len(r.boxes) == 0: continue
        for box in r.boxes:
            x1, y1, x2, y2 = [round(float(v)) for v in box.xyxy[0]]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_w, x2), min(img_h, y2)
            bw, bh = x2-x1, y2-y1
            if bw <= 0 or bh <= 0: continue
            cid = int(box.cls[0])
            dets.append({
                'id':         det_id,
                'bbox':       {'x': x1, 'y': y1, 'width': bw, 'height': bh},
                'class_id':   cid,
                'label':      CLASSES[cid] if cid < 39 else 'unknown',
                'shape':      ID_TO_SHAPE.get(cid, 'unknown'),
                'chromia':    ID_TO_CHROMIA.get(cid, 'unknown'),
                'size':       compute_size(bw, bh),
                'confidence': round(float(box.conf[0]), 4),
            })
            det_id += 1
    return dets

def draw_overlay(img_bgr, detections):
    vis = img_bgr.copy()
    for det in detections:
        b = det['bbox']
        x, y, x2, y2 = b['x'], b['y'], b['x']+b['width'], b['y']+b['height']
        color = SHAPE_COLOR.get(det['shape'], DEFAULT_COLOR)
        cv2.rectangle(vis, (x, y), (x2, y2), color, 1)
        s_abbr = det['shape'][:3].upper()
        c_abbr = det['chromia'][:4]
        sz     = det['size'][0]
        label  = f"{s_abbr}/{c_abbr}/{sz} {det['confidence']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.28, 1)
        ty = max(y - 2, th + 2)
        cv2.rectangle(vis, (x, ty-th-2), (x+tw, ty+2), (0, 0, 0), -1)
        cv2.putText(vis, label, (x, ty), cv2.FONT_HERSHEY_SIMPLEX,
                    0.28, color, 1, cv2.LINE_AA)

    legend_items = [
        ('NOR=Normocyte',  (0,   255,   0)),
        ('OVA=Ovalocyte',  (255, 255,   0)),
        ('ECH=Echinocyte', (0,   165, 255)),
        ('TAR=Target',     (255, 200,   0)),
        ('SPH=Spherocyte', (50,  255, 100)),
        ('SCH=Schistocyte',(0,   128, 255)),
    ]
    lx = vis.shape[1] - 205
    ly_start = vis.shape[0] - len(legend_items) * 17 - 6
    cv2.rectangle(vis, (lx-14, ly_start-4), (vis.shape[1]-2, vis.shape[0]-2),
                  (0, 0, 0), -1)
    for i, (text, lc) in enumerate(legend_items):
        ly = ly_start + i * 17
        cv2.rectangle(vis, (lx-10, ly-10), (lx, ly), lc, -1)
        cv2.putText(vis, text, (lx+4, ly), cv2.FONT_HERSHEY_SIMPLEX,
                    0.31, lc, 1, cv2.LINE_AA)
    return vis

def infer_image(model, img_path):
    img = cv2.imread(str(img_path))
    if img is None: raise ValueError(f'Cannot read: {img_path}')
    h, w = img.shape[:2]
    t0         = time.perf_counter()
    results    = model.predict(img, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, verbose=False)
    latency_ms = (time.perf_counter() - t0) * 1000
    dets = parse_detections(results, w, h)
    output = {
        'image_id':       Path(img_path).name,
        'image_size':     {'width': w, 'height': h},
        'model':          Path(BEST_PT).stem,
        'latency_ms':     round(latency_ms, 2),
        'num_detections': len(dets),
        'avg_confidence': round(sum(d['confidence'] for d in dets)/len(dets), 3) if dets else 0.0,
        'detections':     dets,
    }
    return output, img, latency_ms

print('Inference helpers ready')

In [ ]:
# Load FP32 .pt — fastest on RTX 3050 laptop
infer_model = YOLO(BEST_PT)
print(f'Loaded: {BEST_PT}')

In [ ]:
RUN_TAG     = Path(BEST_PT).parent.parent.name   # e.g. 'yolo11n_1024' or 'yolo11n_1024-4'
INFER_OUT   = INFER_OUT.parent / f'rbc_test_output_{RUN_TAG}'
OVERLAY_DIR = INFER_OUT / 'overlays'
JSON_OUT    = INFER_OUT / 'json'
INFER_OUT.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(exist_ok=True)
JSON_OUT.mkdir(exist_ok=True)

test_images = sorted((DATASET_DIR / 'images' / 'test').glob('*.jpg'))
print(f'Running inference on {len(test_images)} test images  [imgsz={IMGSZ}]')
print(f'Output -> {INFER_OUT}   (new folder per trained run — never overwrites a previous run)\n')

latencies   = []
stats_log   = []
CHECKPOINTS = {1, 10, 20}

for idx, img_path in enumerate(test_images, 1):
    output, img_bgr, ms = infer_image(infer_model, img_path)
    latencies.append(ms)

    # Save JSON
    with open(JSON_OUT / (img_path.stem + '_pred.json'), 'w') as f:
        json.dump(output, f, indent=2)

    # Save overlay image
    overlay = draw_overlay(img_bgr, output['detections'])
    cv2.imwrite(str(OVERLAY_DIR / (img_path.stem + '_overlay.jpg')), overlay)

    print(f'  [{idx:3d}/{len(test_images)}] {img_path.name:<22s}  '
          f'{output["num_detections"]:3d} dets  {ms:.1f} ms', flush=True)

    if idx in CHECKPOINTS:
        arr = np.array(latencies)
        s = {
            'label':     f'After {idx} image(s)',
            'mean_ms':   round(float(arr.mean()), 2),
            'median_ms': round(float(np.median(arr)), 2),
            'min_ms':    round(float(arr.min()), 2),
            'max_ms':    round(float(arr.max()), 2),
        }
        stats_log.append(s)
        print(f'\n  -- {s["label"]} --  median={s["median_ms"]}ms  mean={s["mean_ms"]}ms\n')

# ── Final summary (exclude cold-start image 1) ────────────────────────────────
lats = np.array(latencies[1:])
final = {
    'label':          f'All {len(test_images)} images (excl. cold start)',
    'n_images':       len(test_images),
    'mean_ms':        round(float(lats.mean()), 2),
    'median_ms':      round(float(np.median(lats)), 2),
    'min_ms':         round(float(lats.min()), 2),
    'max_ms':         round(float(lats.max()), 2),
    'pct_under_10ms': round(float((lats < 10).mean() * 100), 1),
}
stats_log.append(final)

print(f'\n{"="*65}')
print(f'  LATENCY SUMMARY  (imgsz={IMGSZ}, excl. cold start)')
print(f'{"="*65}')
print(f'  {"Checkpoint":<38s}  {"Median":>8s}  {"Mean":>8s}  {"Min":>7s}')
print(f'  {"-"*63}')
for s in stats_log:
    pct = f'  <10ms: {s.get("pct_under_10ms","?")}%' if 'pct_under_10ms' in s else ''
    print(f'  {s["label"]:<38s}  {s["median_ms"]:>7.1f}ms  '
          f'{s["mean_ms"]:>7.1f}ms  {s["min_ms"]:>6.1f}ms{pct}')

with open(INFER_OUT / 'latency_stats.json', 'w') as f:
    json.dump(stats_log, f, indent=2)
print(f'\n  Overlays -> {OVERLAY_DIR}')
print(f'  JSONs    -> {JSON_OUT}')
print(f'  Stats    -> {INFER_OUT}/latency_stats.json')

## 9b. Contour / Polygon Extraction  (YOLO11 Segmentation)

> **Requires a `-seg` model** (e.g. `yolo11n-seg.pt`) trained on polygon-format labels.
> `infer_model` above is `yolo11n.pt` — box-only, its results carry no `.masks`.
>
> Once a segmentation model exists at `SEG_BEST_PT` below, `result.masks.xy` /
> `result.masks.xyn` return per-instance contour coordinates directly from the
> Ultralytics results object — no `cv2.findContours` needed at inference time.
>
> The benchmark cell below times the model forward pass (`detect_ms`) and the
> polygon-extraction step (`contour_ms`) **separately** across a batch of test
> FOVs, so a slow result can be attributed to the model itself vs. the contour
> method — same target as `rbc_contour_test.ipynb`'s heuristic-contour
> benchmark, for direct comparison once a `-seg` checkpoint exists.

In [ ]:
def parse_detections_with_contours(yolo_results, img_w, img_h):
    """
    Same schema as parse_detections(), plus a 'contour' field per detection
    when the model is a segmentation model (result.masks is not None):
      contour.points_norm -> [[x,y], ...] normalized 0-1  (result.masks.xyn)
      contour.points_px   -> [[x,y], ...] pixel coords    (result.masks.xy)
    Falls back to box-only output (no 'contour' key) for a detection-only model.

    NOTE: r.masks.xy / r.masks.xyn recompute polygons for ALL instances on
    every access (not just one) — they are read ONCE per image here and
    indexed, not called inside the per-box loop, or cost becomes O(N^2)
    for N instances.
    """
    dets, det_id = [], 1
    for r in yolo_results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        has_masks = getattr(r, 'masks', None) is not None
        polys_px  = r.masks.xy  if has_masks else None
        polys_nrm = r.masks.xyn if has_masks else None
        for i, box in enumerate(r.boxes):
            x1, y1, x2, y2 = [round(float(v)) for v in box.xyxy[0]]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_w, x2), min(img_h, y2)
            bw, bh = x2 - x1, y2 - y1
            if bw <= 0 or bh <= 0:
                continue
            cid = int(box.cls[0])
            det = {
                'id':         det_id,
                'bbox':       {'x': x1, 'y': y1, 'width': bw, 'height': bh},
                'class_id':   cid,
                'label':      CLASSES[cid] if cid < 39 else 'unknown',
                'shape':      ID_TO_SHAPE.get(cid, 'unknown'),
                'chromia':    ID_TO_CHROMIA.get(cid, 'unknown'),
                'size':       compute_size(bw, bh),
                'confidence': round(float(box.conf[0]), 4),
            }
            if has_masks:
                poly_px  = polys_px[i]                   # (N, 2) float32 pixel coords
                poly_nrm = polys_nrm[i]                  # (N, 2) float32 normalized coords
                det['contour'] = {
                    'points_px':   [[round(float(px), 1), round(float(py), 1)] for px, py in poly_px],
                    'points_norm': [[round(float(nx), 6), round(float(ny), 6)] for nx, ny in poly_nrm],
                }
            dets.append(det)
            det_id += 1
    return dets


def draw_contour_overlay(img_bgr, detections):
    """Draws filled-alpha polygons for detections that carry a 'contour' (falls back to bbox)."""
    vis = img_bgr.copy()
    overlay = img_bgr.copy()
    for det in detections:
        color = SHAPE_COLOR.get(det['shape'], DEFAULT_COLOR)
        contour = det.get('contour')
        if contour and contour['points_px']:
            pts = np.array(contour['points_px'], dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], color)
            cv2.polylines(vis, [pts], isClosed=True, color=color, thickness=1, lineType=cv2.LINE_AA)
        else:
            b = det['bbox']
            cv2.rectangle(vis, (b['x'], b['y']), (b['x']+b['width'], b['y']+b['height']), color, 1)
    return cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)


def seg_infer_image(model, img_path):
    """Same shape as infer_image() in Section 9, but times the model forward
    pass (detect_ms, includes mask-branch + per-instance mask assembly) and
    the polygon-extraction step (contour_ms, r.masks.xy/xyn conversion)
    SEPARATELY — so a slow FOV can be attributed to the model or to contour
    extraction instead of only seeing one combined number."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f'Cannot read: {img_path}')
    h, w = img.shape[:2]

    t0 = time.perf_counter()
    results = model.predict(img, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMGSZ, verbose=False)
    detect_ms = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    dets = parse_detections_with_contours(results, w, h)
    contour_ms = (time.perf_counter() - t1) * 1000

    n_with_contour = sum(1 for d in dets if 'contour' in d)
    timing = {'detect_ms': round(detect_ms, 2), 'contour_ms': round(contour_ms, 2),
              'total_ms': round(detect_ms + contour_ms, 2)}
    output = {
        'image_id':        Path(img_path).name,
        'image_size':      {'width': w, 'height': h},
        'model':            Path(SEG_BEST_PT).stem,
        'num_detections':   len(dets),
        'num_with_contour': n_with_contour,
        **timing,
        'detections':       dets,
    }
    return output, img, timing


print('Contour extraction helpers ready (parse_detections_with_contours, draw_contour_overlay, seg_infer_image)')

In [ ]:
# ── Point this at a trained -seg checkpoint once one exists, e.g.:
#   PROJECT_DIR / 'yolo11n-seg_1024' / 'weights' / 'best.pt'
SEG_RUN_NAME = f"{Path(BASE_MODEL).stem}-seg_{IMGSZ}" if not Path(BASE_MODEL).stem.endswith('-seg') \
               else f"{Path(BASE_MODEL).stem}_{IMGSZ}"
SEG_BEST_PT  = PROJECT_DIR / SEG_RUN_NAME / 'weights' / 'best.pt'
if not SEG_BEST_PT.exists():
    seg_candidates = sorted(PROJECT_DIR.glob('*seg*/weights/best.pt'), key=lambda p: p.stat().st_mtime)
    if seg_candidates:
        SEG_BEST_PT = seg_candidates[-1]

TARGET_MS = 200

if SEG_BEST_PT.exists():
    SEG_RUN_TAG     = SEG_BEST_PT.parent.parent.name   # unique per training run, e.g. 'yolo11n-seg_1024'
    SEG_INFER_OUT   = Path('F:/Livo/Data - 2026/Rbc') / f'rbc_seg_test_output_{SEG_RUN_TAG}'
    SEG_OVERLAY_DIR = SEG_INFER_OUT / 'overlays'
    SEG_JSON_OUT    = SEG_INFER_OUT / 'json'
    SEG_INFER_OUT.mkdir(parents=True, exist_ok=True)
    SEG_OVERLAY_DIR.mkdir(exist_ok=True)
    SEG_JSON_OUT.mkdir(exist_ok=True)

    seg_model = YOLO(str(SEG_BEST_PT))
    print(f'Loaded: {SEG_BEST_PT}')
    print(f'Output -> {SEG_INFER_OUT}   (new folder per trained run — never overwrites a previous run)\n')

    N_RUN = min(20, len(test_images))   # raise to len(test_images) to cover all 118
    print(f'Running seg inference + contour timing on {N_RUN} test images  [imgsz={IMGSZ}]\n')

    seg_timing_log = []
    for idx, img_path in enumerate(test_images[:N_RUN], 1):
        output, img_bgr, timing = seg_infer_image(seg_model, img_path)
        seg_timing_log.append({
            'image': img_path.name, 'n_dets': output['num_detections'],
            'n_contours': output['num_with_contour'], **timing,
        })

        with open(SEG_JSON_OUT / (img_path.stem + '_pred.json'), 'w') as f:
            json.dump(output, f, indent=2)

        overlay = draw_contour_overlay(img_bgr, output['detections'])
        cv2.imwrite(str(SEG_OVERLAY_DIR / (img_path.stem + '_overlay.jpg')), overlay)

        flag = '' if timing['total_ms'] < TARGET_MS else '  <-- OVER 200ms BUDGET'
        print(f"  [{idx:3d}/{N_RUN}] {img_path.name:<22s}  {output['num_detections']:3d} dets  "
              f"{output['num_with_contour']:3d} contours  detect={timing['detect_ms']:7.1f}ms  "
              f"contour={timing['contour_ms']:7.1f}ms  total={timing['total_ms']:7.1f}ms{flag}")

    # ── Summary (exclude cold-start image 1) — split by stage to localize the
    #    bottleneck: high detect_ms -> model/GPU issue; high contour_ms -> mask
    #    decode / polygon-extraction issue. ────────────────────────────────────
    rows = seg_timing_log[1:] if len(seg_timing_log) > 1 else seg_timing_log
    detect_arr  = np.array([r['detect_ms']  for r in rows])
    contour_arr = np.array([r['contour_ms'] for r in rows])
    total_arr   = np.array([r['total_ms']   for r in rows])

    print(f'\n{"="*70}')
    print(f'  SEG TIMING SUMMARY  (n={len(rows)}, excl. cold start, target <{TARGET_MS}ms/FOV)')
    print(f'{"="*70}')
    print(f'  {"Stage":<12s} {"Median":>8s} {"Mean":>8s} {"Min":>8s} {"Max":>8s}')
    for name, arr in [('Detect', detect_arr), ('Contour', contour_arr), ('Total', total_arr)]:
        print(f'  {name:<12s} {np.median(arr):7.1f}ms {arr.mean():7.1f}ms {arr.min():7.1f}ms {arr.max():7.1f}ms')

    pct_under = (total_arr < TARGET_MS).mean() * 100
    print(f'\n  {pct_under:.1f}% of FOVs under {TARGET_MS}ms  (avg {np.mean([r["n_dets"] for r in rows]):.0f} RBCs/FOV)')
    bottleneck = 'DETECT (model)' if detect_arr.mean() > contour_arr.mean() else 'CONTOUR (mask decode)'
    print(f'  Dominant cost: {bottleneck}')

    with open(SEG_INFER_OUT / 'seg_timing_stats.json', 'w') as f:
        json.dump(seg_timing_log, f, indent=2)
    print(f'\n  Overlays -> {SEG_OVERLAY_DIR}')
    print(f'  JSONs    -> {SEG_JSON_OUT}')
    print(f'  Stats    -> {SEG_INFER_OUT}/seg_timing_stats.json')
else:
    print(f'No segmentation model found at {SEG_BEST_PT}')
    print('Train one first: set BASE_MODEL = "yolo11n-seg.pt" and rerun Section 6 (train_model),')
    print('with dataset labels in YOLO-seg polygon format (not the box-only labels from Section 4).')
    print('Then rerun this cell for per-instance contour extraction + timing benchmark.')

## 10. PDF Results Report

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec

PDF_PATH   = Path('F:/Livo/Data - 2026/1753802296/rbc_results_report.pdf')
STATS_1024 = Path('F:/Livo/Data - 2026/1753802296/rbc_test_output/latency_stats.json')
STATS_640  = Path('F:/Livo/Data - 2026/1753802296/rbc_test_output_640/latency_stats.json')

C1, C2   = '#1a6eb5', '#e05c1a'   # blue=1024, orange=640
CBG      = '#f0f4f8'
CHEADER  = '#1e3a5f'
CALT     = '#dce8f5'
PAGE_W, PAGE_H = 11, 8.5

plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,
                     'axes.spines.right':False,'axes.grid':True,
                     'grid.alpha':0.3,'grid.linestyle':'--'})

with open(STATS_1024) as f: s1024 = json.load(f)
with open(STATS_640)  as f: s640  = json.load(f)

cold_1024 = s1024[0]['mean_ms']
cold_640  = s640['checkpoints'][0]['mean_ms']
n_test    = s640['all_images']['n_images']
all_640   = s640['all_images']
all_640['label'] = f'All {all_640["n_images"]} images (excl. cold start)'
total_1024 = round(cold_1024 + s1024[-1]['mean_ms'] * (n_test - 1), 1)
total_640  = round(cold_640  + all_640['mean_ms']   * (n_test - 1), 1)

def make_table(ax, data, title):
    ax.axis('off')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=10, loc='left', color=CHEADER)
    n_cols = len(data[0])
    colors = []
    for r in range(len(data)):
        if r == 0:   colors.append([CHEADER]*n_cols)
        elif r%2==1: colors.append(['#ffffff']*n_cols)
        else:        colors.append([CALT]*n_cols)
    tbl = ax.table(cellText=data, cellLoc='center', loc='center', cellColours=colors)
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 1.7)
    for col in range(n_cols): tbl[0, col].set_text_props(color='white', fontweight='bold')
    for (r,c), cell in tbl.get_celld().items(): cell.set_edgecolor('#c5d5e8'); cell.set_linewidth(0.5)

def build_rows(stats, cold, n_total):
    rows = [['Checkpoint','Mean (ms)','Median (ms)','Min (ms)','Max (ms)','Total (ms)']]
    for s in stats:
        pct = f'  <10ms: {s["pct_under_10ms"]}%' if 'pct_under_10ms' in s else ''
        if 'After' in s['label']:
            count = int(s['label'].split()[1])
            tot   = round(cold + s['mean_ms'] * (count-1), 1) if count > 1 else round(cold, 1)
        else:
            tot = round(cold + s['mean_ms'] * (n_total-1), 1)
        rows.append([s['label']+pct, str(s['mean_ms']), str(s['median_ms']),
                     str(s['min_ms']), str(s['max_ms']), f'{tot} ms'])
    return rows

rows_1024 = build_rows(s1024, cold_1024, n_test)
rows_640  = build_rows(s640['checkpoints'] + [all_640], cold_640, n_test)

def add_labels(ax, bars, fmt='{:.1f}ms', pad=0.3, fs=7.5):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+pad, fmt.format(h),
                ha='center', va='bottom', fontsize=fs, fontweight='bold', color='#2c3e50')

def styled(ax, title, ylabel, ylim=None):
    ax.set_title(title, fontsize=10, fontweight='bold', color=CHEADER, pad=8)
    ax.set_ylabel(ylabel, fontsize=9, color='#555'); ax.tick_params(labelsize=8.5)
    ax.set_facecolor('#fafcff')
    if ylim: ax.set_ylim(ylim)

def header(fig, title, sub):
    fig.patches.append(mpatches.FancyBboxPatch((0,0.91),1,0.09,transform=fig.transFigure,
        boxstyle='square,pad=0',facecolor=CHEADER,zorder=0))
    fig.text(0.5,0.955,title,ha='center',va='center',fontsize=15,fontweight='bold',color='white')
    fig.text(0.5,0.918,sub,  ha='center',va='center',fontsize=9, color='#aad4f5')

with PdfPages(str(PDF_PATH)) as pdf:

    # ── Page 1: Latency Tables ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(PAGE_W,PAGE_H), facecolor=CBG)
    header(fig,'RBC Morphology Detection — Inference Latency Results',
           'YOLO11n  |  118 Test FOV Images (1600x1096 px)  |  GPU: NVIDIA RTX 3050')
    fig.text(0.05,0.895,
        'Cold start = first-image GPU warm-up (always high).   Median = stable per-image speed.',
        fontsize=8,color='#666',style='italic')
    gs = GridSpec(2,1,figure=fig,top=0.87,bottom=0.07,hspace=0.55)
    make_table(fig.add_subplot(gs[0]), rows_1024, 'imgsz = 1024   —   Trained & inferred at 1024x1024')
    make_table(fig.add_subplot(gs[1]), rows_640,  'imgsz = 640    —   Trained & inferred at 640x640')
    fig.text(0.5,0.02,
        'Mean = average (skewed by cold start)   |   Median = real per-image speed   |   '
        'Total = cumulative time for all 118 images',ha='center',fontsize=7.5,color='#888')
    pdf.savefig(fig,bbox_inches='tight'); plt.close()

    # ── Page 2: Charts ──────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(PAGE_W,PAGE_H), facecolor=CBG)
    header(fig,'RBC Morphology Detection — Visual Comparison','imgsz=1024  vs  imgsz=640   |   118 Test FOV Images')
    w = 0.32; x = np.arange(3)

    ax1 = fig.add_axes([0.06,0.54,0.40,0.33])
    b1 = ax1.bar(x-w/2,[13.4,14.6,12.6],w,label='imgsz=1024',color=C1,alpha=0.88,zorder=3)
    b2 = ax1.bar(x+w/2,[10.5,11.4, 9.2],w,label='imgsz=640', color=C2,alpha=0.88,zorder=3)
    ax1.axhline(10,color='red',linestyle='--',linewidth=1.5,label='10ms Target',zorder=4)
    ax1.set_xticks(x); ax1.set_xticklabels(['Median','Mean','Min'],fontsize=9)
    ax1.legend(fontsize=8,loc='upper right')
    add_labels(ax1,b1,pad=0.2); add_labels(ax1,b2,pad=0.2)
    styled(ax1,'Per-Image Latency (ms)','Time (ms)',ylim=(0,22))

    ax2 = fig.add_axes([0.57,0.54,0.38,0.33])
    bars2 = ax2.bar(['imgsz=1024','imgsz=640'],[total_1024,total_640],
                    color=[C1,C2],alpha=0.88,width=0.38,zorder=3)
    for bar,val in zip(bars2,[total_1024,total_640]):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
                 f'{val} ms\n({val/1000:.2f} s)',ha='center',va='bottom',
                 fontsize=9,fontweight='bold',color='#2c3e50')
    styled(ax2,f'Total Time — All {n_test} Images','Time (ms)',ylim=(0,max(total_1024,total_640)*1.3))

    ax3 = fig.add_axes([0.06,0.10,0.40,0.33])
    bars3 = ax3.bar(['imgsz=1024','imgsz=640'],[0,35],color=[C1,C2],alpha=0.88,width=0.38,zorder=3)
    ax3.axhline(100,color='green',linestyle='--',linewidth=1.5,label='100% Goal',zorder=4)
    for bar,val in zip(bars3,[0,35]):
        ax3.text(bar.get_x()+bar.get_width()/2,val+1.5,f'{val}%',
                 ha='center',va='bottom',fontsize=12,fontweight='bold',color='#2c3e50')
    ax3.legend(fontsize=8); styled(ax3,'% Images Under 10ms Target','% of Images',ylim=(0,120))

    ax4 = fig.add_axes([0.57,0.10,0.38,0.33])
    b3 = ax4.bar(x-w/2,[0.520,0.797,0.490],w,label='imgsz=1024',color=C1,alpha=0.88,zorder=3)
    b4 = ax4.bar(x+w/2,[0.391,0.657,0.422],w,label='imgsz=640', color=C2,alpha=0.88,zorder=3)
    ax4.set_xticks(x); ax4.set_xticklabels(['mAP50','Precision','Recall'],fontsize=9)
    ax4.legend(fontsize=8)
    add_labels(ax4,b3,fmt='{:.3f}',pad=0.008,fs=7.5); add_labels(ax4,b4,fmt='{:.3f}',pad=0.008,fs=7.5)
    styled(ax4,'Accuracy Metrics','Score (0-1)',ylim=(0,1.05))
    pdf.savefig(fig,bbox_inches='tight'); plt.close()

    # ── Page 3: Comparison Table ────────────────────────────────────────────────
    fig = plt.figure(figsize=(PAGE_W,PAGE_H), facecolor=CBG)
    header(fig,'RBC Morphology Detection — Model Comparison',
           'imgsz=1024  vs  imgsz=640   |   All 118 Test FOV Images')
    summary = [
        ['Metric',                    'imgsz = 1024',     'imgsz = 640'],
        ['Model Architecture',        'YOLO11n',          'YOLO11n'],
        ['Training Batch Size',       '4',                '8'],
        ['mAP50',                     '0.520',            '0.391'],
        ['Precision',                 '0.797',            '0.657'],
        ['Recall',                    '0.490',            '0.422'],
        ['Avg Detections / FOV',      '~250',             '~230'],
        ['Median Latency (per image)','13.4 ms',          '10.5 ms'],
        ['Mean Latency (per image)',  '14.6 ms',          '11.4 ms'],
        ['Min Latency (per image)',   '12.6 ms',          '9.2 ms'],
        ['% Images Under 10ms',       '0%',               '35%'],
        ['Total — All 118 Images',   f'{total_1024} ms', f'{total_640} ms'],
        ['NMS Overhead',              'Negligible',       'High (136 ms on val)'],
    ]
    make_table(fig.add_axes([0.05,0.20,0.90,0.68]), summary, 'Head-to-Head Comparison')
    defs = [
        'mAP50 — Detection accuracy at 50% bounding box overlap (higher is better)',
        'Precision — Of all predicted cells, how many were correct',
        'Recall — Of all real RBC cells, how many did the model detect',
        'Median Latency — Stable per-image speed, unaffected by cold-start spike',
        'NMS Overhead — Time to filter duplicate overlapping boxes (more raw boxes = more time)',
    ]
    for i, d in enumerate(defs):
        fig.text(0.06, 0.17-i*0.033, f'• {d}', fontsize=8, color='#444')
    pdf.savefig(fig,bbox_inches='tight'); plt.close()

print(f'PDF saved -> {PDF_PATH}')

## Notes & Findings

**Model:** YOLO11n · 39 classes (13 shapes × 3 chromia) · trained at imgsz=1024

**Speed (RTX 3050 laptop, FP32 .pt):**
- Median: **13.4 ms/FOV** (stable after warm-up)
- Cold start: ~600–1000 ms (first image only — GPU initialization)
- TensorRT FP16 tested → **slower** (20.4ms) due to thermal throttle under sustained load
- `half=True` tested → **slower** (17.7ms) — RTX 3050 laptop does not benefit from PyTorch FP16
- To reach <10ms: requires desktop GPU (RTX 3080+) or batch processing (4 FOVs simultaneously → ~3-4ms/FOV amortized)

**Accuracy:**
- mAP50=0.520 · Precision=0.797 · Recall=0.490
- Only 3 of 13 shape classes have training data (normocytes, ovalocytes, echinocytes)
- Other 10 shapes (acanthocytes, sickle cells, schistocytes, etc.) need pathologist-annotated examples

**Size classification:**
- Microcyte / Normocyte / Macrocyte derived from bbox diameter — not a trained class
- Calibrate `MICROCYTE_MAX=35` and `MACROCYTE_MIN=55` for your microscope's microns/pixel ratio

**Next steps with real labels:**
1. Annotate rare shape types → activate empty classes → retrain
2. Chromia validation from pixel HSV values inside each detected bbox
3. Batch inference for throughput (process 4 FOVs at once)